In [1]:
# spikeinterface

import spikeinterface.full as si
from spikeinterface.preprocessing import bandpass_filter, common_reference, phase_shift
import spikeinterface.widgets as sw

from spikeinterface.sorters import installed_sorters
from spikeinterface.sorters import get_default_sorter_params, get_sorter_params_description

from spikeinterface.sorters import run_sorter as ss
from spikeinterface.sorters import Kilosort4Sorter as ks

from spikeinterface.curation import apply_sortingview_curation  # optional
from spikeinterface.postprocessing import compute_principal_components
import spikeinterface.qualitymetrics as sqm

# probe
from probeinterface.plotting import plot_probe
from probeinterface import write_probeinterface, read_probeinterface
from probeinterface import ProbeGroup, write_prb, read_prb

# kilosort
import kilosort
from kilosort import io
from kilosort import run_kilosort, DEFAULT_SETTINGS
from kilosort.io import load_probe

# curation
import spikeinterface.curation as sc
# pip install huggingface_hub ## if not already installed in your environment
import huggingface_hub # import the classifier repo

# view traces
import ephyviewer

# extras
import shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import re
import os
import json
from itertools import product
import datetime # Get the current time 
import psutil # forces to kill processes

# sanity check for versions and availability of sorters
print("SpikeInterface version:", si.__version__)
print("Kilosort version:", kilosort.__version__)
print("Huggingface Hub version:", huggingface_hub.__version__)
# check which sorters are installed and get kilosort4 default parameters and descriptions
print(installed_sorters())
# kilosort4 is available as a sorter in spikeinterface, but it is not installed by default. You need to install it separately and make sure it is in your system path for spikeinterface to find it. If you have it installed, you can check its version and get its default parameters and descriptions as follows:
# print(ss.get_sorter_version("kilosort4"))
params = get_default_sorter_params(sorter_name_or_class='kilosort4')
print("Parameters:\n", params)

desc = get_sorter_params_description(sorter_name_or_class='kilosort4')
print("Descriptions:\n", desc)

# check widget backend
# sw.set_default_plotter_backend(backend="ipywidgets")
sw.set_default_plotter_backend(backend="matplotlib")
# activate the widget backend for interactive plotting in Jupyter notebooks
# %matplotlib widget
print(sw.get_default_plotter_backend())

SpikeInterface version: 0.103.2
Kilosort version: 4.1.5
Huggingface Hub version: 1.4.1
['kilosort4', 'simple', 'spykingcircus2', 'tridesclous2']
Parameters:
 {'batch_size': 60000, 'nblocks': 1, 'Th_universal': 9, 'Th_learned': 8, 'nt': 61, 'shift': None, 'scale': None, 'batch_downsampling': 1, 'artifact_threshold': inf, 'nskip': 25, 'whitening_range': 32, 'highpass_cutoff': 300, 'binning_depth': 5, 'sig_interp': 20, 'drift_smoothing': [0.5, 0.5, 0.5], 'nt0min': None, 'dmin': None, 'dminx': 32, 'min_template_size': 10, 'template_sizes': 5, 'nearest_chans': 10, 'nearest_templates': 100, 'max_channel_distance': 32, 'max_peels': 100, 'templates_from_data': True, 'n_templates': 6, 'n_pcs': 6, 'Th_single_ch': 6, 'acg_threshold': 0.2, 'ccg_threshold': 0.25, 'cluster_neighbors': 10, 'cluster_downsampling': 20, 'max_cluster_subset': 25000, 'x_centers': None, 'cluster_init_seed': 5, 'duplicate_spike_ms': 0.25, 'position_limit': 100, 'do_CAR': True, 'invert_sign': False, 'save_extra_vars': False,

In [18]:
root_directory  = os.path.normpath(r"I:\Neuropixels_Backup\IDO1")
dest_path       = os.path.normpath(r'X:\IDO1') 
now             = datetime.datetime.now() # Format the time as YYYYMMDDHHMM 
formatted_time  = now.strftime('%Y%m%d%H%M')

# Initialize a dictionary to store the 1st and 2nd level directories
folders         = {1: []}

# Walk through the directory tree
for dirpath, dirnames, filenames in os.walk(root_directory):
    
    # Calculate the level (relative to root_directory)
    level = os.path.normpath(dirpath).replace(root_directory, "").strip(os.sep).count(os.sep) + 1

    # Store only the first two levels
    if level in folders:
        folders[level].extend([os.path.join(dirpath, d) for d in dirnames])

max_depth = max(len(path.split(os.sep)) for path in folders[1])
folders2 = [path for path in folders[1] if len(path.split(os.sep)) == max_depth]

# filter only ####-##-##_##-##-## folders using regexp
pattern = r"\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}"
folders2 = [path for path in folders2 if re.search(pattern, os.path.basename(path))]
print(folders2)

['I:\\Neuropixels_Backup\\IDO1\\20260219_ET1_rec1\\2026-02-19_15-55-40', 'I:\\Neuropixels_Backup\\IDO1\\20260219_ET1_rec1\\2026-02-20_12-44-43']


In [19]:
# import spikeinterface.extractors as se

folder_path = r"I:\Neuropixels_Backup\IDO1\20260219_ET1_rec1\2026-02-19_15-55-40"

# List available streams
streams = si.get_neo_streams('openephysbinary', folder_path)
blocks = si.get_neo_num_blocks('openephysbinary', folder_path)
print("Available blocks:", blocks)
print("Available streams:", streams)
# using regexp extract all the 'Probe' type (ProbeA, ProbeB) from streams (e.g. 'Record Node 101#Neuropix-PXI-100.ProbeA')
stream_names, stream_ids = streams
probes = [re.search(r'Probe\w+', n).group() for n in stream_names]
print("Probes found:", probes)
print("Available streams:", stream_ids)
dest_path_rec           = os.path.join(dest_path, os.path.basename(folder_path), probes[0])
print("Destination path for recording:", dest_path_rec)
recName                 = os.path.basename(os.path.normpath(folder_path))
mainDir                 = os.path.basename(os.path.dirname(folder_path))
print("Recording name:", recName)
print("Main directory:", mainDir)

Available blocks: 1
Available streams: (['Record Node 101#Neuropix-PXI-100.ProbeA', 'Record Node 101#Neuropix-PXI-100.ProbeB'], ['0', '1'])
Probes found: ['ProbeA', 'ProbeB']
Available streams: ['0', '1']
Destination path for recording: X:\IDO1\2026-02-19_15-55-40\ProbeA
Recording name: 2026-02-19_15-55-40
Main directory: 20260219_ET1_rec1


In [ ]:
reRunKilosort = False  # if True it will re-run kilosort even if the folder exists
reRunLFP      = False  # if True it will re-run LFP processing even if the folder exists
min_duration  = 600    # minimum duration in seconds (10 minutes)

for dataPath in folders2:
    # dataPath = r"I:\Neuropixels_Backup\IDO1\20260219_ET1_rec1\2026-02-19_15-55-40" # for testing
    dataPath = os.path.normpath(dataPath)
    print(f"\nProcessing: {dataPath}")
    
    # grab some information
    streams = si.get_neo_streams('openephysbinary', dataPath)
    # print("Available streams:", streams)
    # extract all the 'ProbeX' part from the stream_id
    stream_names, stream_ids = streams
    probes = [re.search(r'Probe\w+', n).group() for n in stream_names]
    print("Probes found:", probes)
    # loop through the segments:
    for id_num in range(len(stream_ids)):
        full_raw_rec            = si.read_openephys(dataPath, block_index=0,stream_id=stream_ids[id_num])
        dest_path_rec           = os.path.join(dest_path, os.path.basename(dataPath), probes[id_num])
        if not os.path.exists(dest_path_rec):
            os.makedirs(dest_path_rec, exist_ok=True)
            
        fs                      = full_raw_rec.get_sampling_frequency()
        num_segments            = full_raw_rec.get_num_segments()
        durations               = [full_raw_rec.get_num_samples(seg) / fs for seg in range(num_segments)]
        print(f"Durations (s): {durations}")

        # Skip if shorter than 10 minutes (600 seconds)
        valid_segments = [seg for seg in range(num_segments) 
                        if (full_raw_rec.get_num_samples(seg) / fs) >= min_duration]

        # if not valid_segments:
        #     print(f"Skipping {dataPath}: no valid segments ≥ {min_duration}s")
        #     continue

        full_raw_rec            = full_raw_rec.select_segments(valid_segments) # keep only valid segments
        fs                      = full_raw_rec.get_sampling_frequency()
        Chan_IDs                = full_raw_rec.channel_ids
        num_chan                = full_raw_rec.get_num_channels()
        chan_pos                = full_raw_rec.get_channel_locations()
        recName                 = os.path.basename(os.path.normpath(dataPath))
        mainDir                 = os.path.basename(os.path.dirname(dataPath))
        dtype                   = full_raw_rec.get_dtype()
        gain_value              = full_raw_rec.get_channel_gains()    # µV per count
        offset_value            = full_raw_rec.get_channel_offsets()  # offset in µV

        print(full_raw_rec)

        # probe = ...  # From SpikeInterface tutorial, or recording.get_probe()
        probe                   = full_raw_rec.get_probe()
        pg                      = ProbeGroup() # Multiple probes can be added to a ProbeGroup. We only have one, but a ProbeGroup wrapper is still necessary for `write_prb` to work.
        pg.add_probe(probe)
        probeNameKS             = recName + '_' + probes[id_num] +'_KS.prb'
        probePathKS             = os.path.join(dest_path_rec,probeNameKS)
        write_prb(probePathKS, pg)

        probeNameSI             = recName + '_' + probes[id_num] +'_SI.json'
        probePathSI             = os.path.join(dest_path_rec,probeNameSI)
        write_probeinterface(probePathSI, probe) # this won't work for KS4, but still useful for SI

        # -------------------------------
        # LFP Processing
        # -------------------------------
        lfp_folder    = os.path.join(dest_path_rec, 'lfp')
        os.makedirs(lfp_folder, exist_ok=True)

        lfp_raw_files = [f for f in os.listdir(lfp_folder) if f.endswith('.raw')]

        if reRunLFP or not lfp_raw_files:
            # 1. Bandpass filter to LFP band from the raw recording (0.5 – 300 Hz)
            recording_lfp = si.bandpass_filter(recording=full_raw_rec, freq_min=0.5, freq_max=300)
            # 2. Downsample to 1250 Hz
            recording_lfp = si.resample(recording_lfp, resample_rate=1250)
            # 3. Rescale from raw counts to µV (float32) using the recording's gain and offset
            recording_lfp = si.scale(recording_lfp, gain=gain_value, offset=offset_value, dtype='float32')

            # Quick preview before saving — random 30 s window from segment 0
            lfp_dur_preview = recording_lfp.get_num_samples(segment_index=0) / 1250
            win             = 30
            rng             = np.random.default_rng()
            t_start         = float(rng.uniform(0, max(0, lfp_dur_preview - win)))
            t_end           = t_start + win
            print(f"LFP preview: {t_start:.1f} – {t_end:.1f} s")
            sw.plot_traces(
                recording_lfp,
                mode="map",
                time_range=(t_start, t_end),
                order_channel_by_depth=True,
                show_channel_ids=False,
                clim=(-100, 100),   # µV colour scale — adjust if needed
            )
            plt.title(f"LFP  {t_start:.1f}–{t_end:.1f} s  |  {recName}  {probes[id_num]}")
            plt.tight_layout()
            plt.show()

            # 4. Save as binary .raw — n_jobs=1 avoids multiprocessing pickling overhead on Windows
            #    which causes hangs with complex lazy chains (filter → resample → scale)
            recording_lfp_saved = recording_lfp.save(
                folder=lfp_folder,
                n_jobs=1,
                chunk_duration='10s',
                overwrite=True
            )
            print(f"LFP saved to {lfp_folder}")
        else:
            # Load existing LFP binary (already in µV, float32)
            lfp_raw_path        = os.path.join(lfp_folder, lfp_raw_files[0])
            recording_lfp_saved = si.read_binary(
                lfp_raw_path,
                sampling_frequency=1250,
                num_channels=num_chan,
                dtype='float32',
                channel_ids=Chan_IDs,
                gain_to_uV=np.ones(num_chan),    # already in µV — no further scaling needed
                offset_to_uV=np.zeros(num_chan)
            )
            recording_lfp_saved = recording_lfp_saved.set_probe(probe)
            print(f"Loaded existing LFP from {lfp_raw_path}")

        print(recording_lfp_saved)

        # -------------------------------
        # Spike-band preprocessing
        # -------------------------------
        recording_shift         = phase_shift(recording=full_raw_rec)
        recording_cmr           = common_reference(recording=recording_shift, operator="median")
        recording_f = bandpass_filter(
            recording=recording_cmr,
            freq_min=500,
            freq_max=6000
        )

        # -------------------------------
        # 2️⃣ Define preprocessed folder (directly on destination drive)
        # -------------------------------
        preprocesseDir = os.path.join(dest_path_rec, 'preprocessed')
        os.makedirs(preprocesseDir, exist_ok=True)

        # -------------------------------
        # 3️⃣ Save or load binary recording
        # -------------------------------
        raw_files = [f for f in os.listdir(preprocesseDir) if f.endswith('.raw')]

        if not raw_files:
            # Save new preprocessed binary
            recording_saved = recording_f.save(
                folder=preprocesseDir,
                n_jobs=-1,
                chunk_duration='1s',
                overwrite=True
            )
        else:
            # Load existing binary
            # Find the first .raw file (should be only one)
            raw_file_path = os.path.join(preprocesseDir, raw_files[0])
            recording_saved = si.read_binary(
                raw_file_path,
                sampling_frequency=fs,
                num_channels=num_chan,
                dtype=dtype,
                channel_ids=Chan_IDs,
                gain_to_uV=gain_value,
                offset_to_uV=offset_value
            )
            recording_saved = recording_saved.set_probe(probe)
            print(f"Loaded existing binary recording from {raw_file_path}")

        # -------------------------------
        # 4️⃣ Kilosort4 via SpikeInterface
        # -------------------------------
        kilosort_folder = os.path.join(preprocesseDir, "kilosort4")

        if not (reRunKilosort is False and os.path.exists(kilosort_folder)):

            sorting = si.run_sorter(
                sorter_name="kilosort4",
                recording=recording_saved,
                folder=kilosort_folder,   # <-- use "folder", not output_folder
                remove_existing_folder=True,
                verbose=True,
                whitening_range=32,
                dminx=32,
                nblocks=5,
                do_CAR=False, # prefer cmr
                )

        else:
            print(f"Skipping Kilosort: {kilosort_folder} already exists")
            sorting = si.read_sorter_folder(kilosort_folder)

        # -------------------------------
        # 5️⃣ SortingAnalyzer for automated curation
        # -------------------------------
        analyzer_folder = os.path.join(kilosort_folder, "analyzer")
        phy_folder      = os.path.join(kilosort_folder, "phy_export")

        if not os.path.exists(analyzer_folder):
            
            analyzer = si.create_sorting_analyzer(
                sorting=sorting,
                recording=recording_saved,
                format="memory",
                # format="binary_folder",
                # folder=analyzer_folder,
                # overwrite=True,
                folder=None,
            )
            # qualityM = sqm.get_default_qm_params()
            # metric_names = list(qualityM.keys())
            # metrics = sqm.compute_quality_metrics(analyzer)
            # metrics with more control:
            analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=600, seed=2205)
            analyzer.compute("waveforms", ms_before=1.3, ms_after=2.6, n_jobs=-1)
            analyzer.compute("templates", operators=["average", "median", "std"])
            analyzer.compute("template_metrics", include_multi_channel_metrics=True)
            analyzer.compute("template_metrics", metric_names=["spread", "exp_decay", "velocity_above", "velocity_below"])
            analyzer.compute("principal_components", n_components=3, mode="by_channel_global", whiten=True, n_jobs=-1)
            analyzer.compute(['noise_levels','spike_locations','spike_amplitudes','correlograms','quality_metrics',
                            'unit_locations','template_similarity'],
                            n_jobs=-1)
            ### do auto curation
                
            # export to Phy
            si.export_to_phy(
                analyzer,
                output_folder=phy_folder,
                copy_binary=False,
                n_jobs=-1 # use all available cores. Set to 1 to disable parallel copying.
                )

            # Apply the noise/not-noise model
            noise_neuron_labels = sc.auto_label_units(
                sorting_analyzer=analyzer,
                repo_id="SpikeInterface/UnitRefine_noise_neural_classifier",
                trust_model=True,
                export_to_phy=True # we will export to phy after both models have been applied, so we can do it in one step at the end
            )
            noise_units = noise_neuron_labels[noise_neuron_labels['prediction']=='noise']
            analyzer_neural = analyzer.remove_units(noise_units.index)

            # Apply the sua/mua model
            sua_mua_labels = sc.auto_label_units(
                sorting_analyzer=analyzer_neural,
                repo_id="SpikeInterface/UnitRefine_sua_mua_classifier",
                trust_model=True,
                export_to_phy=True
            )
            
            analyzer.save_as(folder=analyzer_folder,
                            format="binary_folder")
            # save the labels as a csv for easier access and visualization
            all_labels = pd.concat([sua_mua_labels, noise_units]).sort_index()
            all_labels.to_csv(os.path.join(analyzer_folder, "unit_labels.csv"))
        else:
            print(f"Analyzer already exists at {analyzer_folder}, loading existing analyzer")
            analyzer = si.load_sorting_analyzer(analyzer_folder)
            labels_path = os.path.join(analyzer_folder, "unit_labels.csv") # load labels from csv
            if os.path.exists(labels_path):
                all_labels = pd.read_csv(labels_path, index_col=0)
            else:
                print(f"Labels file not found at {labels_path}, unable to load labels")
                all_labels = None
            
        print(analyzer)
        # all_labels = pd.concat([sua_mua_labels, noise_units]).sort_index()
        # print(all_labels)

### Plot with ephyviewer

In [ ]:
# plot 5 seconds starting at 10 s
# w_ts = sw.plot_traces(full_raw_rec, mode="map", time_range=(5, 6), show_channel_ids=False, order_channel_by_depth=True)
# si.plot_traces(full_raw_rec, mode="map", time_range=(5, 6), show_channel_ids=False, order_channel_by_depth=True)
sig_source = ephyviewer.SpikeInterfaceRecordingSource(recording=recording_f)
app = ephyviewer.mkQApp()
win = ephyviewer.MainViewer(debug=True, show_auto_scale=True)

view = ephyviewer.TraceViewer(source=sig_source, name='signals')
win.add_view(view)
win.show() 
app.exec()